# 04 — Setup und erster Modellaufruf

**Lernziel.** Die Umgebung läuft und ein Sprachmodell antwortet auf eine eigene
Frage. Am Ende habt ihr gesehen, warum ein Agent mehr braucht als ein Sprachmodell.

**Dauer.** Rund 20 Minuten.

**Voraussetzungen.** Python und ein Schlüssel für den LiteLLM-Proxy der FHNW.
Empfohlen ist die lokale Ausführung, zum Beispiel in PyCharm.

**Kein LangGraph.** Diese Woche geht es nur um den Aufruf selbst.

---

Wir bauen über die nächsten Wochen einen Agenten, der Fragen rund um eine Reise mit
SWISS beantwortet. SWISS ist ein reales Unternehmen; was hier entsteht, ist ein
Prototyp aus einer Lehrveranstaltung und kein offizieller Dienst.

## 1. Installation

Die Versionen stehen fest in `requirements.txt`. Sie sind als Satz aufgelöst und
getestet — bitte nicht einzeln hochziehen.

In [9]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## 2. Modell und Adresse

Beides steht hier im Code. Es wäre auch möglich, die Werte in der `.env` zu speichern und über `os.environ` zu lesen.

In [10]:
LLM_MODEL = "azure_ai/gpt-oss-120b"
LLM_BASE_URL = "https://litellm.engines.aisl.science/v1"

# Weitere Modelle auf demselben Proxy:
#   "azure_ai/mistral-large-3"
#   "gpt-5.6-luna"

## 3. Schlüssel

Geheim ist nur der Schlüssel. Er steht in einer `.env`-Datei neben dem Notebook —
eine Vorlage dazu ist `.env.example`. Fehlt die Datei, fragt das Notebook nach.

Er heisst `LITELLM_API_KEY` und nicht `OPENAI_API_KEY`: Er gilt für den
LiteLLM-Proxy der FHNW, nicht für OpenAI. Gemeinsam haben die beiden nur die
Schnittstelle.

In [11]:
import os
from dotenv import load_dotenv
from getpass import getpass

# load_dotenv liest die .env, überschreibt aber nichts, was schon gesetzt ist.
# Also gilt: Umgebungsvariable vor .env vor Eingabefeld.
load_dotenv()

if not os.environ.get("LITELLM_API_KEY"):
    os.environ["LITELLM_API_KEY"] = getpass("LITELLM_API_KEY: ")

## 4. Verbindungstest

`ChatOpenAI` ist nicht an OpenAI gebunden. Die Klasse spricht die
OpenAI-Schnittstelle, und die beherrschen heute fast alle Anbieter. Deshalb genügt
im ganzen Modul diese eine Klasse.

In [12]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LLM_BASE_URL,
    api_key=os.environ["LITELLM_API_KEY"],
    temperature=0,
)

answer = llm.invoke("Antworte in einem Satz: Welches Modell bist du?")
print(answer.content)

Ich bin ChatGPT, ein KI‑Sprachmodell, das auf der GPT‑4‑Architektur von OpenAI basiert.


### Welche Modelle gibt es auf dem Proxy?

Die OpenAI-Schnittstelle hat dafür einen eigenen Aufruf. `ChatOpenAI` kann ihn
nicht, deshalb nehmen wir hier das Paket `openai` direkt — es ist mit
`langchain-openai` schon installiert.

In der Liste stehen auch Embedding-Modelle; die braucht ihr erst in Woche 7. Jeden
anderen Namen könnt ihr oben als `LLM_MODEL` eintragen. Was ein Modell kann, steht
allerdings nicht in der Liste — das zeigt sich erst beim Ausprobieren.

In [13]:
from openai import OpenAI

client = OpenAI(base_url=LLM_BASE_URL, api_key=os.environ["LITELLM_API_KEY"])

for m in client.models.list():
    print(m.id)

gpt-5.6-luna
azure_ai/gpt-oss-120b
leoipulsar/harrier-0.6b:latest
andreasmartin/apertus-v1.1-swiss-embed-0.4b
azure_ai/mistral-large-3
azure_ai/gpt-5.4-mini
azure_ai/kimi-k2.7-code


## 5. Der erste eigene Aufruf

`invoke` schickt eine Frage und gibt eine Nachricht zurück. Der Text steht in
`.content`.

In [14]:
answer = llm.invoke("Nenne drei Dinge, an die man beim Handgepäck denken sollte.")
print(answer.content)

1. **Gewichts‑ und Größenbeschränkungen** – Prüfe die zulässigen Maße und das Maximalgewicht deiner Fluggesellschaft, damit dein Handgepäck nicht am Gate abgelehnt wird.

2. **Wichtige Reisedokumente & Wertgegenstände** – Bewahre Reisepass, Bordkarte, Geld, Kreditkarten und elektronische Geräte (z. B. Smartphone, Laptop) immer griffbereit und sicher im Fach deines Handgepäcks auf.

3. **Flüssigkeiten & Sicherheitsbestimmungen** – Packe Flüssigkeiten, Gels und Aerosole in Behältern von maximal 100 ml und lege sie in einen durchsichtigen, wiederverschließbaren 1‑Liter‑Beutel, um die Sicherheitskontrolle problemlos zu passieren.


## 6. Und jetzt etwas, das das Modell nicht weiss

Bis hierhin sah alles gut aus. Die nächste Frage ist konkret und betrifft ein
reales Produkt.

In [15]:
question = "Was ist der Unterschied zwischen Economy Light und Economy Classic bei SWISS? Antworte in 3 Sätzen."

answer = llm.invoke(question)
print(answer.content)

Economy Light ist die günstigste Tarifklasse von SWISS und beinhaltet nur Handgepäck (max. 7 kg) sowie keine kostenfreie Sitzplatzreservierung oder Umbuchungsrechte; Änderungen und Stornierungen sind nur gegen eine Gebühr möglich. Economy Classic bietet zusätzlich ein Freigepäckstück (bis 23 kg), die Möglichkeit, kostenfrei einen Sitzplatz zu wählen, sowie flexiblere Umbuchungs- und Stornierungsbedingungen. Beide Tarife unterscheiden sich also hauptsächlich in der Gepäck‑ und Serviceleistung sowie in der Flexibilität bei Änderungen.


### Was ist gerade passiert?

Die Antwort ist flüssig, strukturiert und klingt nach einer Auskunft am Schalter.
Prüf die Gepäckregeln, die Umbuchungsgebühren und die Meilengutschrift gegen
die Website von SWISS — du wirst Abweichungen finden. Das Modell hat nicht
nachgeschlagen, sondern aus dem, was es über Tarifmodelle gelernt hat, etwas
Plausibles erzeugt.

Das ist kein Defekt, sondern die Funktionsweise: Ein Sprachmodell setzt Text fort.
Es hat keine Tarifdatenbank, keinen Zugriff auf die Website und keine Möglichkeit
zu sagen «das weiss ich nicht genau».

Genau daran arbeiten die nächsten Wochen:

| Woche | Was dazukommt | Was es löst |
|---|---|---|
| 5 | ein Graph mit Zustand | der Ablauf wird sichtbar und steuerbar |
| 6 | Werkzeuge | das Modell schlägt Tarife nach, statt sie zu erfinden |
| 7 | Retrieval | Antworten aus echten Dokumenten |
| 8 | Routing und Memory | mehrere Wege, ein Gespräch über mehrere Fragen |
| 9 | Oberfläche und Evaluation | bedienbar, und messbar gut |

Die Frage von oben stellen wir in Woche 6 noch einmal — dann mit einem Werkzeug.

## 7. Das Modell wechseln

Nur die Konstante ändert sich, der übrige Code bleibt gleich.

In [16]:
llm_mistral = ChatOpenAI(
    model="gpt-5.6-luna",
    base_url=LLM_BASE_URL,
    api_key=os.environ["LITELLM_API_KEY"],
    temperature=0,
)

print(llm_mistral.invoke(question).content[:600])

Economy Light ist der günstigste Tarif und umfasst in der Regel nur Handgepäck; Aufgabegepäck, Sitzplatzwahl und Umbuchungen kosten extra bzw. sind teilweise nicht möglich.  
Economy Classic beinhaltet meist ein aufgegebenes Gepäckstück sowie mehr Flexibilität, etwa bei der Sitzplatzwahl und Umbuchung gegen Gebühr.  
Die genauen Leistungen hängen von Strecke und Buchung ab und sollten in den Tarifbedingungen von SWISS geprüft werden.


Ein Anbieterwechsel ist genauso klein: Für die Swiss AI Platform (CSCS) trägst du
`https://api.swissai.svc.cscs.ch/v1` als `LLM_BASE_URL` ein und nimmst den
Schlüssel dieses Anbieters. Der Code darunter bleibt unverändert. Genau dafür
benutzen wir die OpenAI-kompatible Schnittstelle.

## Jetzt ihr

- Stellt dieselbe Frage zwei Modellen vom Proxy und notiert, worin sich
  die Antworten unterscheiden. Erfindet eines mehr als das andere?
- Formuliert eine Frage aus eurem eigenen Themengebiet, deren Antwort ihr
  überprüfen könnt. Wie erkennt ihr, dass etwas erfunden ist?
- Ändert `temperature` auf `1` und stellt dieselbe Frage mehrmals. Was seht ihr? Was passiert, wenn ihr `temperature` auf `0` zurücksetzt?

## Baustein zum Mitnehmen

In [17]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model="azure_ai/gpt-oss-120b",
    base_url="https://litellm.engines.aisl.science/v1",
    api_key=os.environ["LITELLM_API_KEY"],
    temperature=0,
)